# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities (record sets, fields, columns) are referenced by their Croissant `@id` fields for reproducibility and clarity.

### Dataset Source
- Croissant schema: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load the dataset metadata and prepare for record extraction using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the croissant dataset
dataset = mlc.Dataset(croissant_url)

# Get the metadata object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review record sets, their `@id`s, and their respective field `@id`s and names. This helps you select which record sets or fields to load in the next steps.

For every entity, we'll show the value of its `@id`, which uniquely identifies that record set or field in the schema.

In [ ]:
# List available record sets and their fields by @id
print('Available record sets with fields:')
for record_set in dataset.record_sets:
    print(f"- Record Set Name: {record_set.name}, @id: {record_set.id}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - Field Name: {field.name}, @id: {field.id}, DataType: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction

Find the main analytical table, then extract its data into a DataFrame for analysis.

Below we load all tabular record sets by their `@id`.

In [ ]:
# Identify the main record set(s). For this dataset, use the main data table's @id.
# Inspecting the printout above, we note the relevant record set(s).
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Load each record set into a DataFrame using its @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for Record Set @id: {record_set_id}, rows: {len(records)}")
    else:
        print(f"No records found for Record Set @id: {record_set_id}")

# Show columns of the main table (assume first record set is main for demonstration)
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in the main record set (@id: {main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head(5))

## 4. Exploratory Data Analysis (EDA)

We will now process and explore the data. Example operations:
- Filter records based on a numeric field (e.g., Age).
- Normalize that field.
- Group by another field (e.g., Sex or MSI status).

We'll refer to columns by their Croissant schema `@id` (matching DataFrame columns).

In [ ]:
# Identify a numeric and a group field by their @id from previous steps (adjust as needed for your dataset)
df = dataframes.get(main_rs_id)

# Replace field IDs below with actual IDs from your schema/overview as appropriate
# For demonstration, let's search for a likely field for Age and Sex/MSI status
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'Age' in col or 'age' in col:
        numeric_field_id = col
    if 'Sex' in col or 'sex' in col or 'MSI' in col or 'msi' in col:
        group_field_id = col
print(f"Using numeric_field_id: {numeric_field_id} | group_field_id: {group_field_id}")
# If those are not found, please refer to the printed columns in section 3

# Ensure column is numeric
if numeric_field_id is not None and numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Drop NaNs for demonstration
    df_filtered = df.dropna(subset=[numeric_field_id])
    threshold = df_filtered[numeric_field_id].mean()
    filtered_df = df_filtered[df_filtered[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} rows\n")

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"Normalized {numeric_field_id} (sample):")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nMean {numeric_field_id} by {group_field_id}:")
        print(grouped_df)
else:
    print("Could not find a suitable numeric field. Please check your DataFrame columns.")

## 5. Visualization

Lets visualize age (or another numeric field) distribution and group comparison. Please adjust field IDs or names if different from your schema.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot by group
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, you:
- Loaded FAIR^2 clinicopathological colorectal cancer survivor data via its Croissant schema
- Explored record sets, fields, and referenced each by their `@id`
- Performed EDA: simple filtering, normalization, grouping, and created visualizations for insights

This approach can be adapted for any dataset described by a Croissant schema with `mlcroissant`. For domain-specific explorations, examine the `@id`s listed in section 2 and adjust accordingly.